# PyPSA-Eurelectric Analysis

**Works for any network** - uses `n.statistics` module which adapts to whatever components exist.

**Imports existing functions from pypsa-eur scripts:**
- `scripts/make_summary.py` → `calculate_costs`, `calculate_capacities`, `calculate_energy_balance`, `calculate_curtailment`, `calculate_prices`, `calculate_market_values`, `calculate_metrics`, `calculate_nodal_*` (all use `n.statistics` internally)
- `scripts/plot_summary.py` → `preferred_order`
- `scripts/_helpers.py` → `rename_techs`

**Flow visualization:** Uses PyPSA's built-in `n.plot(flow="mean")` with networkx backend.

**Outputs:** Energy mix, electricity mix, capacities, CO2, costs, curtailment, prices, flexibility, transmission (with flow maps), dispatch, market values, capacity factors - all with EU aggregate and country-by-country breakdown, exported to CSV/Excel.

> **Note:** This notebook must be placed in the main PyPSA-Eur model directory (where `scripts/` folder exists) because it uses relative imports from `scripts/make_summary.py`, `scripts/plot_summary.py`, and `scripts/_helpers.py`.

In [ ]:
import sys
sys.path.insert(0, '.')

import logging
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path

# Cartopy for maps (optional)
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    logging.warning("Cartopy not installed - map plots will be skipped")

# Import existing functions from pypsa-eur scripts
from scripts.make_summary import (
    assign_carriers, assign_locations,
    calculate_costs, calculate_capacities, calculate_energy, calculate_energy_balance,
    calculate_nodal_costs, calculate_nodal_capacities, calculate_nodal_energy_balance,
    calculate_capacity_factors, calculate_nodal_capacity_factors,
    calculate_curtailment, calculate_prices, calculate_weighted_prices,
    calculate_market_values, calculate_metrics
)
from scripts.plot_summary import preferred_order, plot_costs, plot_energy, plot_balances
from scripts._helpers import rename_techs

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Style
sns.set_theme("paper", style="whitegrid")
plt.style.use("bmh")

# PyPSA statistics config
pypsa.set_option("params.statistics.round", 3)
pypsa.set_option("params.statistics.drop_zero", False)
pypsa.set_option("params.statistics.nice_names", False)

RESULTS_DIR = Path('results')
OUTPUT_DIR = Path('analysis_output')
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# =============================================================================
# CONFIGURE NETWORK PATH HERE
# =============================================================================
# Provide the path to your solved network file (.nc format)
# Example: NETWORK_PATH = "results/my_run/networks/elec_s_37_lv1.0__Co2L0.7-1H-T-H-B-I-A-dist1_2050.nc"

NETWORK_PATH = None  # <-- SET YOUR NETWORK PATH HERE

# If NETWORK_PATH is None, attempt to find the first available network in results/
if NETWORK_PATH is None:
    networks_found = []
    if RESULTS_DIR.exists():
        for d in RESULTS_DIR.iterdir():
            if d.is_dir() and (d / 'networks').exists():
                for nc in (d / 'networks').glob('*.nc'):
                    networks_found.append(nc)
    if networks_found:
        NETWORK_PATH = networks_found[0]
        logger.info(f"Auto-detected network: {NETWORK_PATH}")
    else:
        raise FileNotFoundError(
            "No networks found. Please set NETWORK_PATH manually, e.g.:\n"
            "NETWORK_PATH = 'results/my_run/networks/elec_s_37_lv1.0__Co2L0.7-1H-T-H-B-I-A-dist1_2050.nc'"
        )
else:
    NETWORK_PATH = Path(NETWORK_PATH)
    if not NETWORK_PATH.exists():
        raise FileNotFoundError(f"Network file not found: {NETWORK_PATH}")

print(f"Using network: {NETWORK_PATH}")

In [ ]:
# Load the network
n = pypsa.Network(str(NETWORK_PATH))

# Use existing helper functions from make_summary.py
assign_carriers(n)
assign_locations(n)

# Load plotting config
with open('config/plotting.default.yaml', 'r') as f:
    plotting_config = yaml.safe_load(f).get('plotting', {})
tech_colors = plotting_config.get('tech_colors', {})

# Update carrier colors
n.carriers.update({"color": tech_colors})
mask = n.carriers.color.isna() | n.carriers.color.eq("")
n.carriers["color"] = n.carriers.color.mask(mask, "lightgrey")

logger.info(f"Loaded: {NETWORK_PATH}")
print(f"Network: {NETWORK_PATH.name}")
print(f"Snapshots: {len(n.snapshots)}, Buses: {len(n.buses)}")
print(f"Generators: {len(n.generators)}, Links: {len(n.links)}, Stores: {len(n.stores)}")

In [ ]:
# Export helper
def export_df(df, name, excel_writer=None):
    """Export DataFrame to CSV and optionally to Excel sheet."""
    csv_path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(csv_path)
    logger.info(f"Exported: {csv_path}")
    if excel_writer is not None:
        df.to_excel(excel_writer, sheet_name=name[:31])
    return df

def get_country(bus):
    """Extract 2-letter country code from bus name."""
    return str(bus)[:2]

# Colors for plotting
colors = n.carriers.set_index("nice_name").color.where(lambda s: s != "", "lightgrey")

# AC buses (needed for price calculations)
ac_buses = n.buses[n.buses.carrier == 'AC'].index

# Renewable generators
re_carriers = ['solar', 'solar-hsat', 'onwind', 'offwind-ac', 'offwind-dc', 'offwind-float']
re_g = n.generators[n.generators.carrier.isin(re_carriers)].index

# Flexibility link indices (battery, H2)
bc = n.links[n.links.carrier == 'battery charger'].index
bd = n.links[n.links.carrier == 'battery discharger'].index
he = n.links[n.links.carrier == 'H2 electrolysis'].index
hf = n.links[n.links.carrier == 'H2 fuel cell'].index

## 1. Energy Mix (All Vectors)

In [ ]:
# 1. ENERGY MIX - Using calculate_energy_balance from make_summary.py
energy_balance = calculate_energy_balance(n)

# Group by bus_carrier for energy vectors
eb_by_vector = energy_balance.groupby('bus_carrier').sum() / 1e6  # TWh
eb_by_vector = eb_by_vector[eb_by_vector.abs() > 0.1]

# Calculate relative shares
total_energy = eb_by_vector.abs().sum()
eb_shares = (eb_by_vector.abs() / total_energy * 100).round(1)

# Create summary DataFrame
energy_mix_df = pd.DataFrame({
    'TWh': eb_by_vector.round(2),
    'Share (%)': eb_shares
}).sort_values('TWh', key=abs, ascending=False)

print("=== ENERGY BY VECTOR (TWh and %) ===")
print(energy_mix_df)
print(f"\nTotal Energy (absolute): {total_energy:.2f} TWh")

# Visualize main vectors
main_vectors = ['AC', 'low voltage', 'H2', 'gas', 'oil', 'heat']
vectors = eb_by_vector[eb_by_vector.index.isin(main_vectors)]
if len(vectors) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    vectors.abs().plot.bar(ax=ax, color='steelblue')
    ax.set_title('Energy by Vector (TWh)')
    ax.set_ylabel('TWh')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'energy_by_vector.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. Electricity Mix

In [ ]:
# 2. ELECTRICITY MIX - Using n.statistics.supply() for proper snapshot weighting
# Filter to only electricity-delivering technologies (AC and low voltage buses)

# Dynamically detect available electricity bus carriers in this network
desired_elec_carriers = ['AC', 'low voltage']
available_carriers = n.buses.carrier.unique()
elec_carriers = [c for c in desired_elec_carriers if c in available_carriers]

if not elec_carriers:
    print("No electricity bus carriers (AC, low voltage) found in network")
    supply_renamed = pd.Series(dtype=float)
    elec_mix_df = pd.DataFrame()
    total_supply = 0
else:
    # Get supply using n.statistics (accounts for snapshot_weightings)
    supply = n.statistics.supply(bus_carrier=elec_carriers) / 1e6  # TWh
    supply = supply[supply.abs() > 0.1]

    # Group by carrier (supply has MultiIndex: component, carrier)
    if isinstance(supply.index, pd.MultiIndex):
        carrier_idx = supply.index.get_level_values('carrier')
        supply_by_carrier = supply.groupby(carrier_idx).sum()
    else:
        supply_by_carrier = supply

    # Rename techs using existing function
    supply_renamed = supply_by_carrier.groupby(supply_by_carrier.index.map(rename_techs)).sum()
    supply_renamed = supply_renamed.reindex(
        preferred_order.intersection(supply_renamed.index).append(
            supply_renamed.index.difference(preferred_order)
        )
    )

    # Calculate relative shares
    total_supply = supply_renamed[supply_renamed > 0].sum()
    supply_shares = (supply_renamed / total_supply * 100).round(1)

    # Create summary DataFrame with TWh and %
    elec_mix_df = pd.DataFrame({
        'TWh': supply_renamed.round(2),
        'Share (%)': supply_shares
    })
    elec_mix_df = elec_mix_df[elec_mix_df['TWh'] > 0].sort_values('TWh', ascending=False)

    print(f"=== ELECTRICITY MIX (TWh and %) - Bus carriers: {elec_carriers} ===")
    print(elec_mix_df)
    print(f"\nTotal Electricity Supply: {total_supply:.2f} TWh")

# Visualize
if len(elec_mix_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    pos = elec_mix_df['TWh'].sort_values(ascending=True)
    c = [tech_colors.get(t, 'lightgrey') for t in pos.index]
    pos.plot.barh(ax=ax, color=c)
    ax.set_title('Electricity Supply by Technology (TWh)')
    ax.set_xlabel('TWh')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'electricity_mix.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No electricity supply data available for plotting")

## 3. Installed Capacity (Electricity)

In [ ]:
# 3. INSTALLED CAPACITY - Using calculate_capacities from make_summary.py
capacities = calculate_capacities(n) / 1e3  # GW
capacities = capacities[capacities > 0.01]

# Capacities has MultiIndex (component, carrier) - extract carrier level
carrier_idx = capacities.index.get_level_values('carrier')
cap_by_carrier = capacities.groupby(carrier_idx).sum()

# Rename and order
cap_renamed = cap_by_carrier.groupby(cap_by_carrier.index.map(rename_techs)).sum()
cap_renamed = cap_renamed.reindex(
    preferred_order.intersection(cap_renamed.index).append(
        cap_renamed.index.difference(preferred_order)
    )
)

# Calculate relative shares
total_cap = cap_renamed.sum()
cap_shares = (cap_renamed / total_cap * 100).round(1)

# Create summary DataFrame
cap_df = pd.DataFrame({
    'GW': cap_renamed.round(2),
    'Share (%)': cap_shares
}).sort_values('GW', ascending=False)

print("=== INSTALLED CAPACITY (GW and %) ===")
print(cap_df)
print(f"\nTotal Capacity: {total_cap:.2f} GW")

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
c = [tech_colors.get(t, 'lightgrey') for t in cap_renamed.index]
cap_renamed.sort_values(ascending=True).plot.barh(ax=ax, color=c)
ax.set_title('Optimal Capacity by Technology (GW)')
ax.set_xlabel('GW')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'installed_capacity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. CO2 Emissions (Energy Sector & Electricity)

In [ ]:
# 4. CO2 EMISSIONS
co2_by_sector = {}

# Electricity generation CO2
if 'co2_emissions' in n.carriers.columns:
    gen_output = n.generators_t.p.sum()
    gen_carriers = n.generators.carrier
    gen_co2_factor = gen_carriers.map(n.carriers['co2_emissions']).fillna(0)
    elec_co2 = (gen_output * gen_co2_factor).groupby(gen_carriers).sum() / 1e6  # Mt
    co2_by_sector['Electricity'] = elec_co2.sum()

    print("=== CO2 FROM ELECTRICITY (Mt) ===")
    elec_co2_pos = elec_co2[elec_co2 > 0].sort_values(ascending=False)
    print(elec_co2_pos.round(3))
    print(f"Total Electricity CO2: {elec_co2.sum():.2f} Mt")

# Heating CO2 (from gas boilers, etc.)
heat_carriers_co2 = ['gas boiler', 'resistive heater', 'oil boiler']
heat_links = n.links[n.links.carrier.isin(heat_carriers_co2)]
if len(heat_links) > 0:
    heat_output = n.links_t.p0[heat_links.index].sum()
    heat_co2_factor = heat_links.carrier.map(n.carriers['co2_emissions']).fillna(0)
    heat_co2 = (heat_output * heat_co2_factor).sum() / 1e6
    co2_by_sector['Heating'] = heat_co2

# Transport CO2 (from ICE vehicles if modeled)
transport_carriers = ['ICE', 'land transport oil']
for tc in transport_carriers:
    tc_loads = n.loads[n.loads.carrier == tc]
    if len(tc_loads) > 0:
        tc_output = n.loads_t.p_set[tc_loads.index].sum().sum()
        tc_co2 = tc_output * n.carriers.loc[tc, 'co2_emissions'] / 1e6 if tc in n.carriers.index else 0
        co2_by_sector['Transport'] = co2_by_sector.get('Transport', 0) + tc_co2

# Industry CO2
ind_carriers = ['process emissions', 'industry']
for ic in ind_carriers:
    ic_links = n.links[n.links.carrier.str.contains(ic, case=False, na=False)]
    if len(ic_links) > 0:
        co2_by_sector['Industry'] = co2_by_sector.get('Industry', 0) + n.links_t.p0[ic_links.index].sum().sum() / 1e6

co2_sectors = pd.Series(co2_by_sector)
co2_sectors = co2_sectors[co2_sectors > 0]

print("\n=== CO2 BY ENERGY SECTOR (Mt) ===")
print(co2_sectors.round(2))
print(f"Total: {co2_sectors.sum():.2f} Mt")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Electricity by technology
if 'elec_co2_pos' in dir() and len(elec_co2_pos) > 0:
    c = elec_co2_pos.index.map(lambda x: n.carriers.color.get(x, 'lightgrey'))
    elec_co2_pos.plot.bar(ax=axes[0], color=c)
    axes[0].set_title('Electricity CO2 by Technology (Mt)')
    axes[0].set_ylabel('Mt CO2')
    axes[0].tick_params(axis='x', rotation=45)

# By sector
if len(co2_sectors) > 0:
    co2_sectors.plot.pie(ax=axes[1], autopct='%1.1f%%')
    axes[1].set_title('CO2 by Energy Sector')
    axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'co2_emissions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. System Costs (by Technology)

In [ ]:
# 5. SYSTEM COSTS - Using calculate_costs from make_summary.py
costs_raw = calculate_costs(n) / 1e9  # B EUR

# Costs has MultiIndex (cost, component, carrier) - get carrier level
# First, sum across components to get by (cost, carrier)
costs_by_cost_carrier = costs_raw.groupby(['cost', 'carrier']).sum()

# Unstack cost level to get DataFrame with carrier index and capital/marginal columns
costs_df = costs_by_cost_carrier.unstack(level='cost').fillna(0)
costs_df['total'] = costs_df.sum(axis=1)
costs_df = costs_df[costs_df['total'] > 0.01].sort_values('total', ascending=False)

# Rename techs (costs_df now has simple carrier index, not MultiIndex)
costs_renamed = costs_df.groupby(costs_df.index.map(rename_techs)).sum()
costs_renamed = costs_renamed.sort_values('total', ascending=False)

print("=== SYSTEM COSTS BY TECHNOLOGY (bn EUR) ===")
print(costs_renamed.round(2))
print(f"\nTotal System Cost: {costs_renamed['total'].sum():.2f} B EUR")

# Visualize by technology
fig, ax = plt.subplots(figsize=(12, 8))
costs_renamed[['capital', 'marginal']].head(20).plot.bar(ax=ax, stacked=True, color=['steelblue', 'coral'])
ax.set_title('System Costs by Technology (B EUR)')
ax.set_ylabel('B EUR')
ax.tick_params(axis='x', rotation=45)
ax.legend(['CAPEX', 'OPEX'])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'system_costs.png', dpi=150, bbox_inches='tight')
plt.show()

# =============================================================================
# EXAMPLE: Grouping Technologies into Sectors
# =============================================================================
# Customize this mapping to match your analysis needs
tech_to_sector = {
    # Electricity generation
    'solar': 'Electricity', 'solar-hsat': 'Electricity', 'onwind': 'Electricity',
    'offwind-ac': 'Electricity', 'offwind-dc': 'Electricity', 'offwind-float': 'Electricity',
    'CCGT': 'Electricity', 'OCGT': 'Electricity', 'nuclear': 'Electricity',
    'coal': 'Electricity', 'lignite': 'Electricity', 'oil': 'Electricity',
    'ror': 'Electricity', 'hydro': 'Electricity', 'PHS': 'Electricity',
    'biomass': 'Electricity', 'biogas': 'Electricity',
    # Storage
    'battery': 'Storage', 'battery charger': 'Storage', 'battery discharger': 'Storage',
    'H2 electrolysis': 'Storage', 'H2 fuel cell': 'Storage', 'H2 Store': 'Storage',
    # Heating
    'gas boiler': 'Heating', 'resistive heater': 'Heating', 'air heat pump': 'Heating',
    'ground heat pump': 'Heating', 'oil boiler': 'Heating', 'biomass boiler': 'Heating',
    'urban central heat': 'Heating', 'residential rural heat': 'Heating',
    # Transport
    'EV charger': 'Transport', 'V2G': 'Transport', 'land transport oil': 'Transport',
    'BEV charger': 'Transport',
    # Transmission
    'AC': 'Transmission', 'DC': 'Transmission', 'H2 pipeline': 'Transmission',
    # Industry
    'H2 for industry': 'Industry', 'low-temperature heat for industry': 'Industry',
}

# Map technologies to sectors (use 'Other' for unmapped techs)
def map_to_sector(tech):
    return tech_to_sector.get(tech, 'Other')

# Aggregate costs by sector
costs_by_sector = costs_renamed.groupby(costs_renamed.index.map(map_to_sector)).sum()
costs_by_sector = costs_by_sector.sort_values('total', ascending=False)

print("\n=== SYSTEM COSTS BY SECTOR (bn EUR) ===")
print(costs_by_sector.round(2))

# Visualize by sector
if len(costs_by_sector) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    costs_by_sector[['capital', 'marginal']].plot.bar(ax=ax, stacked=True, color=['steelblue', 'coral'])
    ax.set_title('System Costs by Sector (B EUR)')
    ax.set_ylabel('B EUR')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(['CAPEX', 'OPEX'])
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'system_costs_by_sector.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Curtailment

In [ ]:
# 6. CURTAILMENT - Using n.statistics.curtailment() as primary method
# The calculate_curtailment(n) function may fail if network lacks AC/low voltage generators

# Get absolute curtailment using n.statistics (most reliable)
curt_abs = n.statistics.curtailment() / 1e6  # TWh
curt_abs = curt_abs[curt_abs > 0]

print("=== CURTAILMENT (TWh) ===")
if len(curt_abs) > 0:
    print(curt_abs.round(3))
    print(f"\nTotal Curtailment: {curt_abs.sum():.2f} TWh")
else:
    print("No curtailment in this network")

# Try to calculate relative curtailment (may not work for all networks)
print("\n=== CURTAILMENT (%) ===")
try:
    curtailment_pct = calculate_curtailment(n)
    curtailment_pct = curtailment_pct[curtailment_pct > 0]
    if len(curtailment_pct) > 0:
        print(curtailment_pct.round(2))
    else:
        print("No curtailment percentage data available")
except Exception as e:
    # calculate_curtailment requires generators with AC/low voltage bus carriers
    # Fall back to manual calculation
    logger.warning(f"calculate_curtailment failed: {e}")
    print("Relative curtailment calculation not available for this network structure")

    # Manual fallback: calculate for renewable generators
    re_carriers_curt = ['solar', 'solar-hsat', 'onwind', 'offwind-ac', 'offwind-dc', 'offwind-float']
    re_gens_curt = n.generators[n.generators.carrier.isin(re_carriers_curt)]
    if len(re_gens_curt) > 0 and len(n.generators_t.p_max_pu.columns.intersection(re_gens_curt.index)) > 0:
        re_idx = re_gens_curt.index.intersection(n.generators_t.p_max_pu.columns)
        p_max_pot = (n.generators_t.p_max_pu[re_idx] * n.generators.loc[re_idx, 'p_nom_opt']).sum().sum()
        p_actual = n.generators_t.p[re_idx].sum().sum()
        if p_max_pot > 0:
            curt_pct_manual = (p_max_pot - p_actual) / p_max_pot * 100
            print(f"Renewable curtailment (manual calc): {curt_pct_manual:.1f}%")

In [ ]:
# Hourly curtailment (uses re_g defined in helpers)
p_max = n.generators_t.p_max_pu[re_g] * n.generators.loc[re_g, 'p_nom_opt']
p_act = n.generators_t.p[re_g]
curt_h = (p_max - p_act).clip(lower=0).sum(axis=1) / 1e3  # GW

pot = p_max.sum().sum() / 1e6  # TWh
act = p_act.sum().sum() / 1e6  # TWh
print(f"RE potential: {pot:.2f} TWh, actual: {act:.2f} TWh")
print(f"Curtailment: {pot-act:.2f} TWh ({(pot-act)/pot*100:.1f}%)")

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(curt_h.index, curt_h.values, alpha=0.7, color='#ff6666')
ax.set_ylabel('GW')
ax.set_title('Hourly Curtailment')
plt.savefig(OUTPUT_DIR / 'curtailment_hourly.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Prices

In [ ]:
# 5. PRICES - Using calculate_prices and calculate_weighted_prices from make_summary.py
avg_prices = calculate_prices(n)
weighted_prices = calculate_weighted_prices(n)

print("=== AVERAGE PRICES BY CARRIER (EUR/MWh) ===")
print(avg_prices.round(2))

print("\n=== LOAD-WEIGHTED PRICES (EUR/MWh) ===")
print(weighted_prices.round(2))

# Also get metrics for price statistics
metrics = calculate_metrics(n)
print("\n=== PRICE METRICS ===")
price_metrics = metrics[metrics.index.str.contains('price')]
print(price_metrics.round(2))

# Get hourly AC prices for duration curve (load-weighted average across buses)
if len(ac_buses) > 0:
    prices_hourly = n.buses_t.marginal_price[ac_buses]
    load_ac = n.loads_t.p_set[n.loads[n.loads.bus.isin(ac_buses)].index]
    weights = load_ac.sum(axis=1)
    lw_price = (prices_hourly.multiply(weights, axis=0).sum(axis=1) / weights).fillna(prices_hourly.mean(axis=1))
else:
    lw_price = pd.Series(0, index=n.snapshots)

print(f"\nLoad-weighted electricity price (EU average): {lw_price.mean():.2f} EUR/MWh")

# === MARGINAL PRICES BY COUNTRY ===
if len(ac_buses) > 0:
    import plotly.graph_objects as go
    
    countries = pd.Series({b: get_country(b) for b in ac_buses})
    avg_price_by_country = prices_hourly.mean().groupby(countries).mean()

    print("\n=== MARGINAL PRICES BY COUNTRY (EUR/MWh) ===")
    print(avg_price_by_country.sort_values(ascending=False).round(2))

    # Time series plot of marginal prices by country (interactive with plotly)
    price_by_country_ts = prices_hourly.T.groupby(countries).mean().T  # Time series by country
    
    fig = go.Figure()
    
    for country in price_by_country_ts.columns:
        country_avg = avg_price_by_country[country]
        # Add time series
        fig.add_trace(go.Scatter(
            x=price_by_country_ts.index,
            y=price_by_country_ts[country],
            mode='lines',
            name=f'{country}',
            opacity=0.7,
            line=dict(width=1)
        ))
        # Add average line for this country
        fig.add_trace(go.Scatter(
            x=[price_by_country_ts.index[0], price_by_country_ts.index[-1]],
            y=[country_avg, country_avg],
            mode='lines',
            name=f'{country} avg: {country_avg:.1f}',
            line=dict(dash='dash', width=2),
            opacity=0.9
        ))
    
    fig.update_layout(
        title='Electricity Marginal Prices by Country (Time Series)',
        xaxis_title='Time',
        yaxis_title='Marginal Price (EUR/MWh)',
        legend=dict(x=1.02, y=1, font=dict(size=9)),
        hovermode='x unified',
        height=600,
        width=1200
    )
    
    fig.show()

In [ ]:
# LMP Map (requires cartopy)
if HAS_CARTOPY and len(ac_buses) > 0:
    avg_p = prices_hourly.mean()
    bus_xy = n.buses.loc[ac_buses, ['x', 'y']].copy()
    bus_xy['price'] = avg_p
    bus_xy = bus_xy.dropna()

    # Use EqualEarth projection directly
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.EqualEarth()})
    bounds = plotting_config.get('map', {}).get('boundaries', [-11, 30, 34, 71])
    ax.set_extent(bounds, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='whitesmoke')

    sc = ax.scatter(bus_xy['x'], bus_xy['y'], c=bus_xy['price'], s=50, cmap='Spectral_r',
                    transform=ccrs.PlateCarree(), edgecolor='k', linewidth=0.3, zorder=10)
    plt.colorbar(sc, ax=ax, label='EUR/MWh', shrink=0.7)
    ax.set_title('Average Nodal Prices (LMP Map)')
    plt.savefig(OUTPUT_DIR / 'lmp_map.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    logger.info("Skipping LMP map (cartopy not available or no AC buses)")

In [ ]:
# 5b. PRICE DURATION CURVE
price_sorted = lw_price.sort_values(ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Duration curve
axes[0].plot(price_sorted.values, 'b-', lw=1.5)
axes[0].axhline(lw_price.mean(), color='red', linestyle='--', label=f'Mean: {lw_price.mean():.1f}')
axes[0].axhline(lw_price.median(), color='green', linestyle='--', label=f'Median: {lw_price.median():.1f}')
axes[0].set_xlabel('Hours (sorted)')
axes[0].set_ylabel('EUR/MWh')
axes[0].set_title('Price Duration Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram
axes[1].hist(lw_price.values, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[1].axvline(lw_price.mean(), color='red', linestyle='--', lw=2, label=f'Mean: {lw_price.mean():.1f}')
axes[1].set_xlabel('EUR/MWh')
axes[1].set_ylabel('Hours')
axes[1].set_title('Price Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'price_duration.png', dpi=150, bbox_inches='tight')
plt.show()

# Price statistics
print("=== WHOLESALE PRICE STATISTICS ===")
print(f"Mean: {lw_price.mean():.2f} EUR/MWh")
print(f"Median: {lw_price.median():.2f} EUR/MWh")
print(f"Std Dev: {lw_price.std():.2f} EUR/MWh")
print(f"Min: {lw_price.min():.2f} EUR/MWh")
print(f"Max: {lw_price.max():.2f} EUR/MWh")
print(f"Hours < 0: {(lw_price < 0).sum()} ({(lw_price < 0).mean()*100:.1f}%)")
print(f"Hours > 100: {(lw_price > 100).sum()} ({(lw_price > 100).mean()*100:.1f}%)")

## 8. Marginal Technology

In [ ]:
# 6. MARGINAL TECHNOLOGY - Identifies price-setting technology hour by hour
if len(ac_buses) > 0:
    avg_price = prices_hourly.mean(axis=1)
else:
    avg_price = pd.Series(0, index=n.snapshots)

mc = n.generators.marginal_cost
carrier = n.generators.carrier

marg = []
for t in n.snapshots:
    p = avg_price[t]
    out = n.generators_t.p.loc[t]
    active = out[out > 0.1].index
    if len(active) == 0:
        marg.append('none')
    else:
        marg.append(carrier[(mc[active] - p).abs().idxmin()])

marg = pd.Series(marg, index=n.snapshots)
print("=== HOURS BY PRICE-SETTING TECHNOLOGY ===")
print(marg.value_counts())

gas_h = marg.isin(['CCGT', 'OCGT']).sum()
fossil_h = marg.isin(['CCGT', 'OCGT', 'coal', 'lignite', 'oil']).sum()
print(f"\nGas setting price: {gas_h} h ({gas_h/len(marg)*100:.1f}%)")
print(f"Fossil setting price: {fossil_h} h ({fossil_h/len(marg)*100:.1f}%)")

## 9. Flexibility & Storage

In [ ]:
# 7a. STORAGE CAPACITY AND ENERGY PROVIDED
# Uses bc, bd, he, hf defined in helpers cell
storage_data = []

# Storage units
if len(n.storage_units) > 0:
    for carrier in n.storage_units.carrier.unique():
        su = n.storage_units[n.storage_units.carrier == carrier]
        p = n.storage_units_t.p[su.index]
        storage_data.append({
            'Technology': carrier,
            'Power_GW': su.p_nom_opt.sum() / 1e3,
            'Energy_GWh': (su.max_hours * su.p_nom_opt).sum() / 1e3,
            'Discharged_TWh': p.clip(lower=0).sum().sum() / 1e6,
            'Charged_TWh': (-p).clip(lower=0).sum().sum() / 1e6
        })

# Battery stores
batt_stores = n.stores[n.stores.carrier == 'battery']
if len(batt_stores) > 0:
    storage_data.append({
        'Technology': 'Battery',
        'Power_GW': n.links.loc[bd, 'p_nom_opt'].sum() / 1e3 if len(bd) > 0 else 0,
        'Energy_GWh': batt_stores.e_nom_opt.sum() / 1e3,
        'Discharged_TWh': n.links_t.p0[bd].sum().sum() / 1e6 if len(bd) > 0 else 0,
        'Charged_TWh': n.links_t.p0[bc].sum().sum() / 1e6 if len(bc) > 0 else 0
    })

# H2 storage
h2_stores = n.stores[n.stores.carrier.isin(['H2', 'H2 Store'])]
if len(h2_stores) > 0:
    storage_data.append({
        'Technology': 'H2 Storage',
        'Power_GW': n.links.loc[he, 'p_nom_opt'].sum() / 1e3 if len(he) > 0 else 0,
        'Energy_GWh': h2_stores.e_nom_opt.sum() / 1e3,
        'Discharged_TWh': n.links_t.p0[hf].sum().sum() / 1e6 if len(hf) > 0 else 0,
        'Charged_TWh': n.links_t.p0[he].sum().sum() / 1e6 if len(he) > 0 else 0
    })

storage_df = pd.DataFrame(storage_data)
if len(storage_df) > 0:
    storage_df['Duration_h'] = storage_df['Energy_GWh'] / storage_df['Power_GW'].replace(0, np.nan)
    storage_df['Cycles'] = storage_df['Discharged_TWh'] * 1e3 / storage_df['Energy_GWh'].replace(0, np.nan)

    print("=== STORAGE CAPACITY AND ENERGY ===")
    print(storage_df.round(2))

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    storage_df.set_index('Technology')['Power_GW'].plot.bar(ax=axes[0], color='steelblue')
    axes[0].set_title('Power Capacity (GW)')
    axes[0].set_ylabel('GW')
    axes[0].tick_params(axis='x', rotation=45)

    storage_df.set_index('Technology')['Energy_GWh'].plot.bar(ax=axes[1], color='coral')
    axes[1].set_title('Energy Capacity (GWh)')
    axes[1].set_ylabel('GWh')
    axes[1].tick_params(axis='x', rotation=45)

    storage_df.set_index('Technology')['Discharged_TWh'].plot.bar(ax=axes[2], color='green')
    axes[2].set_title('Energy Provided (TWh)')
    axes[2].set_ylabel('TWh')
    axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'storage.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No storage found in network")

In [ ]:
# 7. FLEXIBILITY - Detailed breakdown (upward, downward, total)
# Uses bc, bd, he, hf defined in helpers cell
hours = len(n.snapshots)
flex_data = []

# Storage units (PHS, hydro)
if len(n.storage_units) > 0:
    for carrier in n.storage_units.carrier.unique():
        su = n.storage_units[n.storage_units.carrier == carrier]
        p = n.storage_units_t.p[su.index]
        flex_data.append({
            'Technology': carrier,
            'Capacity_GW': su.p_nom_opt.sum() / 1e3,
            'Upward_TWh': p.clip(lower=0).sum().sum() / 1e6,
            'Downward_TWh': (-p).clip(lower=0).sum().sum() / 1e6
        })

# Battery (uses bd, bc from helpers)
if len(bc) > 0:
    flex_data.append({
        'Technology': 'Battery',
        'Capacity_GW': n.links.loc[bd, 'p_nom_opt'].sum() / 1e3,
        'Upward_TWh': n.links_t.p0[bd].sum().sum() / 1e6,
        'Downward_TWh': n.links_t.p0[bc].sum().sum() / 1e6
    })

# H2 electrolysis/fuel cell (uses he, hf from helpers)
if len(he) > 0:
    flex_data.append({
        'Technology': 'H2 System',
        'Capacity_GW': n.links.loc[hf, 'p_nom_opt'].sum() / 1e3 if len(hf) > 0 else 0,
        'Upward_TWh': n.links_t.p0[hf].sum().sum() / 1e6 if len(hf) > 0 else 0,
        'Downward_TWh': n.links_t.p0[he].sum().sum() / 1e6
    })

# DSM / V2G
v2g = n.links[n.links.carrier == 'V2G'].index
if len(v2g) > 0:
    flex_data.append({
        'Technology': 'V2G',
        'Capacity_GW': n.links.loc[v2g, 'p_nom_opt'].sum() / 1e3,
        'Upward_TWh': n.links_t.p0[v2g].sum().sum() / 1e6,
        'Downward_TWh': 0
    })

flex_df = pd.DataFrame(flex_data)
if len(flex_df) > 0:
    flex_df['Total_TWh'] = flex_df['Upward_TWh'] + flex_df['Downward_TWh']
    flex_df['Available_TWh'] = flex_df['Capacity_GW'] * hours / 1e3
    flex_df['Utilization_%'] = (flex_df['Total_TWh'] / flex_df['Available_TWh'] * 100).fillna(0)

    print("=== FLEXIBILITY BY TECHNOLOGY ===")
    print(flex_df.round(2))

    tot_cap = flex_df['Capacity_GW'].sum()
    tot_up = flex_df['Upward_TWh'].sum()
    tot_down = flex_df['Downward_TWh'].sum()
    tot_avail = tot_cap * hours / 1e3
    tot_util = (tot_up + tot_down) / tot_avail * 100 if tot_avail > 0 else 0

    print(f"\n=== TOTALS ===")
    print(f"Flexibility Capacity: {tot_cap:.2f} GW")
    print(f"Upward Flexibility: {tot_up:.2f} TWh")
    print(f"Downward Flexibility: {tot_down:.2f} TWh")
    print(f"Total Flexibility: {tot_up + tot_down:.2f} TWh")
    print(f"Utilization Rate: {tot_util:.1f}%")

    # Curtailment vs Flexibility comparison (if curt_h is defined)
    if 'curt_h' in dir():
        total_curt_twh = curt_h.sum() / 1e3
        total_flex_twh = flex_df['Total_TWh'].sum()
        print(f"\n=== CURTAILMENT vs FLEXIBILITY ===")
        print(f"Total Curtailment: {total_curt_twh:.2f} TWh")
        print(f"Total Flexibility Used: {total_flex_twh:.2f} TWh")
        print(f"Ratio (Curt/Flex): {total_curt_twh/total_flex_twh:.2f}" if total_flex_twh > 0 else "")

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Capacity
    flex_df.set_index('Technology')['Capacity_GW'].plot.bar(ax=axes[0], color='steelblue')
    axes[0].set_title('Flexibility Capacity (GW)')
    axes[0].set_ylabel('GW')
    axes[0].tick_params(axis='x', rotation=45)

    # Up/Down
    flex_df.set_index('Technology')[['Upward_TWh', 'Downward_TWh']].plot.bar(ax=axes[1], color=['green', 'orange'])
    axes[1].set_title('Flexibility Provided (TWh)')
    axes[1].set_ylabel('TWh')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend(['Upward', 'Downward'])

    # Utilization
    flex_df.set_index('Technology')['Utilization_%'].plot.bar(ax=axes[2], color='purple')
    axes[2].set_title('Utilization Rate (%)')
    axes[2].set_ylabel('%')
    axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'flexibility.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    flex_df = pd.DataFrame()
    print("No flexibility assets found in network")

## 10. Transmission, Distribution & Unmet Demand

In [ ]:
# 10. TRANSMISSION - Using n.statistics.transmission
tx = n.statistics.transmission(groupby=False)
print("=== Transmission (TWh) ===")
print((tx / 1e6).round(2))

# Cross-border capacity
lines = n.lines[['bus0', 'bus1', 's_nom_opt']].copy()
lines['c0'] = lines['bus0'].apply(get_country)
lines['c1'] = lines['bus1'].apply(get_country)
lines['GW'] = lines['s_nom_opt'] / 1e3
cross = lines[lines['c0'] != lines['c1']]
cross['pair'] = cross.apply(lambda x: tuple(sorted([x['c0'], x['c1']])), axis=1)
print("\n=== Cross-border AC Capacity (GW) ===")
print(cross.groupby('pair')['GW'].sum().sort_values(ascending=False).head(10).round(2))

# DC links cross-border
dc_links = n.links[n.links.carrier == 'DC']
if len(dc_links) > 0:
    dc = dc_links[['bus0', 'bus1', 'p_nom_opt']].copy()
    dc['c0'] = dc['bus0'].apply(get_country)
    dc['c1'] = dc['bus1'].apply(get_country)
    dc['GW'] = dc['p_nom_opt'] / 1e3
    dc_cross = dc[dc['c0'] != dc['c1']]
    if len(dc_cross) > 0:
        dc_cross['pair'] = dc_cross.apply(lambda x: tuple(sorted([x['c0'], x['c1']])), axis=1)
        print("\n=== Cross-border DC Capacity (GW) ===")
        print(dc_cross.groupby('pair')['GW'].sum().sort_values(ascending=False).head(10).round(2))

In [ ]:
# 10a. POWER FLOW MAP - Using PyPSA's n.plot() with line_flow/link_flow parameters
# Based on scripts/plot_balance_map.py approach

if HAS_CARTOPY and len(ac_buses) > 0 and len(n.lines) > 0:
    # Get transmission flows using n.statistics
    flow = n.statistics.transmission(groupby=False, bus_carrier='AC')

    # Handle reversed flows
    if not flow.empty:
        flow_reversed_mask = flow.index.get_level_values(1).str.contains("reversed")
        flow_reversed = flow[flow_reversed_mask].rename(lambda x: x.replace("-reversed", ""))
        flow = flow[~flow_reversed_mask].subtract(flow_reversed, fill_value=0)

    # Extract line and link flows
    line_flow = flow.get("Line", pd.Series()) / 1e6  # TWh for scaling
    link_flow = flow.get("Link", pd.Series()) / 1e6  # TWh

    # Line/link widths based on capacity
    line_widths = (n.lines.s_nom_opt / n.lines.s_nom_opt.max()).clip(lower=0.1) * 2

    dc_links_idx = n.links[n.links.carrier == 'DC'].index
    link_widths = pd.Series(0.0, index=n.links.index)
    if len(dc_links_idx) > 0:
        max_dc = n.links.loc[dc_links_idx, 'p_nom_opt'].max()
        if max_dc > 0:
            link_widths.loc[dc_links_idx] = (n.links.loc[dc_links_idx, 'p_nom_opt'] / max_dc).clip(lower=0.1) * 2

    # Bus sizes - ONLY for AC buses, scaled properly
    gen_by_bus = n.generators_t.p.sum().groupby(n.generators.bus).sum()
    bus_sizes = pd.Series(0.0, index=n.buses.index)
    ac_gen_buses = gen_by_bus.index.intersection(ac_buses)
    if len(ac_gen_buses) > 0:
        max_gen = gen_by_bus[ac_gen_buses].max()
        if max_gen > 0:
            bus_sizes.loc[ac_gen_buses] = (gen_by_bus[ac_gen_buses] / max_gen).clip(lower=0.05) * 500

    # Line colors based on flow magnitude (for colorbar)
    line_flow_abs = line_flow.abs()
    if len(line_flow_abs) > 0 and line_flow_abs.max() > 0:
        line_colors = line_flow_abs
    else:
        line_colors = 'darkblue'

    # Create figure with proper projection
    fig, ax = plt.subplots(figsize=(12, 10), subplot_kw={'projection': ccrs.EqualEarth()})

    # Set boundaries
    bounds = plotting_config.get('map', {}).get('boundaries', [-11, 32, 34, 72])

    # Plot network with line colors based on flow
    collection = n.plot(
        ax=ax,
        bus_sizes=bus_sizes * 0.002,
        bus_colors='coral',
        line_widths=line_widths,
        line_colors=line_colors,
        link_widths=link_widths,
        link_colors='purple',
        line_flow=line_flow.abs() * 2 if len(line_flow) > 0 else None,
        link_flow=link_flow.reindex(n.links.index).fillna(0).abs() * 2 if len(link_flow) > 0 else None,
        margin=0.2,
        geomap=True,
        geomap_colors={"border": "grey", "coastline": "grey", "land": "whitesmoke", "ocean": "lightblue"},
        boundaries=bounds,
    )

    # Add colorbar for line flow
    if isinstance(line_colors, pd.Series) and len(line_colors) > 0:
        # Get the line collection from the plot (usually index 2)
        if hasattr(collection, '__iter__') and len(collection) > 2:
            sm = plt.cm.ScalarMappable(
                cmap='viridis',
                norm=plt.Normalize(vmin=0, vmax=line_flow_abs.max())
            )
            sm.set_array([])
            cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', pad=0.05, shrink=0.6, aspect=30)
            cbar.set_label('Transmission Flow (TWh)', fontsize=10)

    ax.set_title('Power Network: Transmission Capacity & Flow')
    plt.savefig(OUTPUT_DIR / 'power_flow_map.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Net power flow between countries
    print("\n=== NET POWER FLOW BETWEEN COUNTRIES (TWh) ===")
    flow_matrix = {}

    for line in n.lines.index:
        c0 = get_country(n.lines.loc[line, 'bus0'])
        c1 = get_country(n.lines.loc[line, 'bus1'])
        if c0 != c1:
            pair = (c0, c1)
            flow_twh = n.lines_t.p0[line].sum() / 1e6 if line in n.lines_t.p0.columns else 0
            flow_matrix[pair] = flow_matrix.get(pair, 0) + flow_twh

    for link in dc_links_idx:
        c0 = get_country(n.links.loc[link, 'bus0'])
        c1 = get_country(n.links.loc[link, 'bus1'])
        if c0 != c1:
            pair = (c0, c1)
            flow_twh = n.links_t.p0[link].sum() / 1e6 if link in n.links_t.p0.columns else 0
            flow_matrix[pair] = flow_matrix.get(pair, 0) + flow_twh

    flow_series = pd.Series(flow_matrix).sort_values(key=abs, ascending=False)
    print(flow_series.head(15).round(2))

else:
    logger.info("Skipping power flow map (cartopy not available, no AC buses, or no lines)")

In [ ]:
# 8b. UNMET DEMAND / LOAD SHEDDING
# Check for load shedding generators or unserved energy
load_shed_carriers = ['load', 'load shedding', 'unserved']
load_shed = n.generators[n.generators.carrier.str.contains('|'.join(load_shed_carriers), case=False, na=False)]

unmet_demand = 0
if len(load_shed) > 0:
    unmet_demand = n.generators_t.p[load_shed.index].sum().sum() / 1e6  # TWh

# Also check for slack variables in constraints
total_load = n.loads_t.p_set.sum().sum() / 1e6
total_supply = n.generators_t.p.sum().sum() / 1e6

print("=== UNMET DEMAND ===")
print(f"Total Load: {total_load:.2f} TWh")
print(f"Total Supply: {total_supply:.2f} TWh")
print(f"Load Shedding: {unmet_demand:.2f} TWh")
print(f"Unmet Demand Rate: {unmet_demand/total_load*100:.3f}%")

In [ ]:
# 8c. DISTRIBUTION GRID CAPACITY
# In PyPSA-Eur, distribution is modeled as links connecting low voltage to medium/high voltage
dist_carriers = ['electricity distribution grid', 'distribution', 'low voltage']
dist_links = n.links[n.links.carrier.str.contains('|'.join(dist_carriers), case=False, na=False)]

if len(dist_links) > 0:
    dist_cap = dist_links.p_nom_opt.sum() / 1e3  # GW
    dist_flow = n.links_t.p0[dist_links.index].abs().sum().sum() / 1e6  # TWh

    dist_by_country = pd.DataFrame({
        'capacity_GW': dist_links.p_nom_opt / 1e3,
        'bus': dist_links.bus0
    })
    dist_by_country['country'] = dist_by_country['bus'].apply(get_country)
    dist_country = dist_by_country.groupby('country')['capacity_GW'].sum()

    print("=== DISTRIBUTION GRID ===")
    print(f"Total Distribution Capacity: {dist_cap:.2f} GW")
    print(f"Distribution Flow: {dist_flow:.2f} TWh")
    print("\nBy Country (GW):")
    print(dist_country.sort_values(ascending=False).round(2))
else:
    # Estimate from low-voltage connected assets
    lv_buses = n.buses[n.buses.carrier.str.contains('low', case=False, na=False)].index
    if len(lv_buses) > 0:
        lv_gens = n.generators[n.generators.bus.isin(lv_buses)]
        lv_loads = n.loads[n.loads.bus.isin(lv_buses)]
        est_dist = (lv_gens.p_nom_opt.sum() + n.loads_t.p_set[lv_loads.index].max().sum()) / 1e3
        print(f"Estimated Distribution Need: {est_dist:.2f} GW")
    else:
        print("No explicit distribution grid modeled")

## 11. Dispatch & Day Example

In [ ]:
disp = n.generators_t.p.copy()
disp.columns = n.generators.carrier.values
disp = disp.T.groupby(level=0).sum().T / 1e3
load = n.loads_t.p_set.sum(axis=1) / 1e3

# Week
wk = min(167, len(n.snapshots)-1)
wd = disp.iloc[:wk+1]
wl = load.iloc[:wk+1]

order = ['nuclear', 'lignite', 'coal', 'biomass', 'ror', 'hydro', 'solar', 'solar-hsat',
         'onwind', 'offwind-ac', 'offwind-dc', 'offwind-float', 'CCGT', 'OCGT', 'oil']
cols = [c for c in order if c in wd.columns and wd[c].sum() > 0]

fig, ax = plt.subplots(figsize=(14, 5))
c = n.carriers.color.reindex(cols).fillna('lightgrey')
wd[cols].plot.area(ax=ax, stacked=True, color=c.values, alpha=0.8, linewidth=0)
ax.plot(wl.index, wl.values, 'k-', lw=2, label='Load')
ax.set_ylabel('GW')
ax.set_title('Weekly Dispatch')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 9b. DAY EXAMPLE - Load with flexibility contribution
# Uses bc, bd, re_g defined in helpers cell
# Select a representative day (high RE, mid-year)
day_start = min(24 * 180, len(n.snapshots) - 24)  # ~July
day_slice = slice(day_start, day_start + 24)

load_day = n.loads_t.p_set.sum(axis=1).iloc[day_slice] / 1e3  # GW

# Flexibility contribution
flex_up_day = pd.Series(0.0, index=load_day.index)
flex_down_day = pd.Series(0.0, index=load_day.index)

# Storage units
if len(n.storage_units) > 0:
    su_p = n.storage_units_t.p.iloc[day_slice].sum(axis=1) / 1e3
    flex_up_day = flex_up_day + su_p.clip(lower=0)
    flex_down_day = flex_down_day + (-su_p).clip(lower=0)

# Battery (uses bc, bd from helpers)
if len(bc) > 0 and len(bd) > 0:
    batt_out = n.links_t.p0[bd].iloc[day_slice].sum(axis=1) / 1e3
    batt_in = n.links_t.p0[bc].iloc[day_slice].sum(axis=1) / 1e3
    flex_up_day = flex_up_day + batt_out
    flex_down_day = flex_down_day + batt_in

# Net load (load - RE generation) - uses re_g from helpers
re_gen_day = n.generators_t.p[re_g].iloc[day_slice].sum(axis=1) / 1e3 if len(re_g) > 0 else pd.Series(0, index=load_day.index)
net_load_day = load_day - re_gen_day

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: Load and net load
axes[0].plot(range(24), load_day.values, 'b-', lw=2, label='Total Load')
axes[0].plot(range(24), net_load_day.values, 'r--', lw=2, label='Net Load (Load - RE)')
axes[0].fill_between(range(24), load_day.values, net_load_day.values, alpha=0.3, color='green', label='RE Generation')
axes[0].set_ylabel('GW')
axes[0].set_title(f'Day Example: Load Profile (Day {day_start//24})')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Bottom: Flexibility contribution
axes[1].bar(range(24), flex_up_day.values, color='green', alpha=0.7, label='Flexibility Discharge')
axes[1].bar(range(24), -flex_down_day.values, color='orange', alpha=0.7, label='Flexibility Charge')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('GW')
axes[1].set_title('Flexibility Contribution')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'day_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Market Value & Capacity Factors

In [ ]:
# 10. MARKET VALUE - Using calculate_market_values from make_summary.py
market_values = calculate_market_values(n)

print("=== MARKET VALUES (EUR/MWh) ===")
print(market_values.round(2))

In [ ]:
# 10b. CAPACITY FACTORS - Using calculate_capacity_factors from make_summary.py
cf = calculate_capacity_factors(n)

print("=== CAPACITY FACTORS ===")
print((cf * 100).round(1))

# Visualize RE capacity factors
re_techs = ['solar', 'onwind', 'offwind', 'ror', 'hydro']
re_cf = cf[[c for c in cf.index if any(r in str(c).lower() for r in re_techs)]]
if len(re_cf) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    (re_cf * 100).sort_values().plot.barh(ax=ax, color='steelblue')
    ax.set_xlabel('Capacity Factor (%)')
    ax.set_title('Renewable Capacity Factors')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'capacity_factors.png', dpi=150, bbox_inches='tight')
    plt.show()

## 13. Summary

In [ ]:
# SUMMARY - using values calculated throughout the notebook
supply = n.statistics.supply().drop("Line", errors='ignore')
gen_twh = supply.sum() / 1e6

# Curtailment total
curt_total = n.statistics.curtailment().sum() / 1e6

# Flexibility total (if flex_df was created)
flex_cap_total = 0
flex_util = 0
if 'flex_df' in dir() and len(flex_df) > 0:
    flex_cap_total = flex_df['Capacity_GW'].sum()
    flex_util = flex_df['Utilization_%'].mean()

print("="*50)
print("SUMMARY")
print("="*50)
print(f"Network: {NETWORK_PATH.name}")
print(f"Period: {n.snapshots[0]} to {n.snapshots[-1]}")
print(f"Generation: {gen_twh:.2f} TWh")
print(f"System Cost: {n.objective/1e9:.2f} B EUR")
print(f"Avg Price: {lw_price.mean():.2f} EUR/MWh")
print(f"Curtailment: {curt_total:.2f} TWh")
print(f"Flexibility: {flex_cap_total:.2f} GW ({flex_util:.1f}% util)")

## 14. Country-by-Country Breakdown & Export

In [ ]:
# 14. COUNTRY-BY-COUNTRY BREAKDOWN - Using calculate_nodal_* functions

# Create Excel writer
excel_path = OUTPUT_DIR / f"analysis_{NETWORK_PATH.stem}.xlsx"
writer = pd.ExcelWriter(excel_path, engine='openpyxl')

# Nodal capacities (by location)
nodal_cap = calculate_nodal_capacities(n) / 1e3  # GW
nodal_cap = nodal_cap[nodal_cap > 0.01]
export_df(nodal_cap.unstack().fillna(0), 'capacities_by_location', writer)

# Nodal energy balance
nodal_eb = calculate_nodal_energy_balance(n) / 1e6  # TWh
export_df(nodal_eb.unstack().fillna(0), 'energy_by_location', writer)

# Nodal costs
nodal_costs = calculate_nodal_costs(n) / 1e9  # B EUR
export_df(nodal_costs.unstack().fillna(0), 'costs_by_location', writer)

# Nodal capacity factors
nodal_cf = calculate_nodal_capacity_factors(n)
export_df((nodal_cf * 100).unstack().fillna(0), 'cf_by_location', writer)

print("=== CAPACITIES BY LOCATION (GW) - Top 10 ===")
print(nodal_cap.groupby('location').sum().sort_values(ascending=False).head(10).round(2))

In [ ]:
# --- Prices by country ---
ac_buses = n.buses[n.buses.carrier == 'AC'].index
if len(ac_buses) > 0:
    bus_prices = n.buses_t.marginal_price[ac_buses].mean()
    price_df = pd.DataFrame({
        'avg_price_EUR_MWh': bus_prices,
        'country': [get_country(b) for b in bus_prices.index]
    })
    prices_by_country = price_df.groupby('country')['avg_price_EUR_MWh'].mean()
    export_df(prices_by_country.to_frame(), 'prices_by_country', writer)

    print("\nAverage prices by country (EUR/MWh):")
    print(prices_by_country.sort_values(ascending=False).round(2))

In [ ]:
# --- Load by country ---
load_df = pd.DataFrame({
    'load_TWh': n.loads_t.p_set.sum() / 1e6,
    'bus': n.loads.bus
})
load_df['country'] = load_df['bus'].apply(get_country)
load_by_country = load_df.groupby('country')['load_TWh'].sum()
export_df(load_by_country.to_frame(), 'load_by_country', writer)

print("\nLoad by country (TWh):")
print(load_by_country.sort_values(ascending=False).round(2))

In [ ]:
# --- Storage by country ---
stores = []
# Storage units (PHS, hydro)
if len(n.storage_units) > 0:
    su = pd.DataFrame({
        'capacity_GW': n.storage_units.p_nom_opt / 1e3,
        'energy_GWh': n.storage_units.max_hours * n.storage_units.p_nom_opt / 1e3,
        'carrier': n.storage_units.carrier,
        'bus': n.storage_units.bus
    })
    su['country'] = su['bus'].apply(get_country)
    stores.append(su)

# Stores (battery, H2 storage)
if len(n.stores) > 0:
    st = pd.DataFrame({
        'energy_GWh': n.stores.e_nom_opt / 1e3,
        'carrier': n.stores.carrier,
        'bus': n.stores.bus
    })
    st['country'] = st['bus'].apply(get_country)
    st['capacity_GW'] = 0  # Stores don't have power capacity
    stores.append(st)

if stores:
    storage_df = pd.concat(stores, ignore_index=True)
    storage_by_country = storage_df.pivot_table(
        values=['capacity_GW', 'energy_GWh'],
        index='country',
        columns='carrier',
        aggfunc='sum'
    ).fillna(0)
    export_df(storage_by_country, 'storage_by_country', writer)
    print("\nStorage capacity by country:")
    print(storage_by_country.round(2))

In [ ]:
# --- Heating by sector/country ---
heat_carriers = ['residential rural heat', 'residential urban decentral heat',
                 'services rural heat', 'services urban decentral heat',
                 'urban central heat', 'low-temperature heat for industry']
heat_links = n.links[n.links.carrier.isin(heat_carriers)]

if len(heat_links) > 0:
    heat_df = pd.DataFrame({
        'heat_TWh': n.links_t.p0[heat_links.index].sum() / 1e6,
        'capacity_GW': heat_links.p_nom_opt / 1e3,
        'carrier': heat_links.carrier,
        'bus': heat_links.bus0
    })
    heat_df['country'] = heat_df['bus'].apply(get_country)
    heat_by_country = heat_df.pivot_table(
        values=['heat_TWh', 'capacity_GW'],
        index='country',
        columns='carrier',
        aggfunc='sum'
    ).fillna(0)
    export_df(heat_by_country, 'heating_by_country', writer)

    # Aggregate by sector (residential, services, industry)
    heat_eu = heat_df.groupby('carrier')[['heat_TWh', 'capacity_GW']].sum()
    export_df(heat_eu, 'heating_eu_by_sector', writer)
    print("\nHeating by sector (TWh):")
    print(heat_eu['heat_TWh'].round(2))

In [ ]:
# --- H2 by country/sector ---
h2_prod_carriers = ['H2 electrolysis', 'SMR', 'SMR CC']
h2_use_carriers = ['H2 fuel cell', 'H2 for industry', 'H2 for shipping',
                   'Fischer-Tropsch', 'Sabatier', 'methanolisation']

h2_links = n.links[n.links.carrier.isin(h2_prod_carriers + h2_use_carriers)]
if len(h2_links) > 0:
    h2_df = pd.DataFrame({
        'flow_TWh': n.links_t.p0[h2_links.index].sum() / 1e6,
        'capacity_GW': h2_links.p_nom_opt / 1e3,
        'carrier': h2_links.carrier,
        'bus': h2_links.bus0
    })
    h2_df['country'] = h2_df['bus'].apply(get_country)
    h2_by_country = h2_df.pivot_table(
        values=['flow_TWh', 'capacity_GW'],
        index='country',
        columns='carrier',
        aggfunc='sum'
    ).fillna(0)
    export_df(h2_by_country, 'h2_by_country', writer)

    h2_eu = h2_df.groupby('carrier')[['flow_TWh', 'capacity_GW']].sum()
    export_df(h2_eu, 'h2_eu_by_use', writer)
    print("\nH2 production/use (TWh):")
    print(h2_eu['flow_TWh'].round(2))

In [ ]:
# --- CO2 emissions by country ---
# From generators with emission factors
if 'co2_emissions' in n.carriers.columns:
    gen_co2 = pd.DataFrame({
        'carrier': n.generators.carrier,
        'bus': n.generators.bus,
        'generation_MWh': n.generators_t.p.sum()
    })
    gen_co2['country'] = gen_co2['bus'].apply(get_country)
    gen_co2['co2_t_per_MWh'] = gen_co2['carrier'].map(n.carriers['co2_emissions']).fillna(0)
    gen_co2['co2_Mt'] = gen_co2['generation_MWh'] * gen_co2['co2_t_per_MWh'] / 1e6

    co2_by_country = gen_co2.pivot_table(
        values='co2_Mt',
        index='country',
        columns='carrier',
        aggfunc='sum'
    ).fillna(0)
    export_df(co2_by_country, 'co2_by_country', writer)

    co2_eu = gen_co2.groupby('carrier')['co2_Mt'].sum()
    export_df(co2_eu.to_frame(), 'co2_eu_by_carrier', writer)

    print("\nCO2 emissions by country (Mt):")
    print(co2_by_country.sum(axis=1).sort_values(ascending=False).round(2))
    print(f"\nTotal EU: {co2_by_country.sum().sum():.2f} Mt")

In [ ]:
# --- Curtailment by country ---
re_carriers = ['solar', 'solar-hsat', 'onwind', 'offwind-ac', 'offwind-dc', 'offwind-float']
re_gens = n.generators[n.generators.carrier.isin(re_carriers)]

if len(re_gens) > 0:
    p_max = n.generators_t.p_max_pu[re_gens.index] * re_gens.p_nom_opt
    p_act = n.generators_t.p[re_gens.index]
    curt_gen = (p_max.sum() - p_act.sum()) / 1e6  # TWh

    curt_df = pd.DataFrame({
        'curtailment_TWh': curt_gen,
        'carrier': re_gens.carrier,
        'bus': re_gens.bus
    })
    curt_df['country'] = curt_df['bus'].apply(get_country)
    curt_by_country = curt_df.pivot_table(
        values='curtailment_TWh',
        index='country',
        columns='carrier',
        aggfunc='sum'
    ).fillna(0)
    export_df(curt_by_country, 'curtailment_by_country', writer)

    print("\nCurtailment by country (TWh):")
    print(curt_by_country.sum(axis=1).sort_values(ascending=False).round(2))

In [ ]:
# Export all summary statistics
export_df(calculate_costs(n).to_frame('value'), 'costs', writer)
export_df(calculate_capacities(n).to_frame('value'), 'capacities', writer)
export_df(calculate_energy(n).to_frame('value'), 'energy', writer)
export_df(calculate_energy_balance(n).to_frame('value'), 'energy_balance', writer)
export_df(calculate_capacity_factors(n).to_frame('value'), 'capacity_factors', writer)
export_df(calculate_curtailment(n).to_frame('value'), 'curtailment', writer)
export_df(calculate_prices(n).to_frame('value'), 'prices', writer)
export_df(calculate_weighted_prices(n).to_frame('value'), 'weighted_prices', writer)
export_df(calculate_market_values(n).to_frame('value'), 'market_values', writer)
export_df(calculate_metrics(n).to_frame('value'), 'metrics', writer)

# Close Excel writer
writer.close()
logger.info(f"All exports saved to: {excel_path}")
print(f"\n{'='*50}")
print(f"EXPORTS COMPLETE")
print(f"{'='*50}")
print(f"Excel file: {excel_path}")
print(f"CSV files in: {OUTPUT_DIR}")

## 15. Scenario Comparison (Multi-Network)

In [ ]:
# 15. SCENARIO COMPARISON (Electricity-focused metrics)
# =============================================================================
# To compare multiple scenarios, either:
# 1. Set NETWORK_PATH = None and let auto-discovery find networks
# 2. Or manually specify paths in scenario_paths list below
# =============================================================================

# Collect networks for comparison
scenario_paths = []
if RESULTS_DIR.exists():
    for d in RESULTS_DIR.iterdir():
        if d.is_dir() and (d / 'networks').exists():
            for nc in (d / 'networks').glob('*.nc'):
                scenario_paths.append(nc)
scenario_paths = scenario_paths[:min(6, len(scenario_paths))]  # Limit to 6

if len(scenario_paths) > 1:
    comparison = []
    price_curves = {}

    # Define desired electricity carriers (will be filtered per network)
    desired_elec_carriers = ['AC', 'low voltage']
    re_carriers_comp = ['solar', 'solar-hsat', 'onwind', 'offwind-ac', 'offwind-dc',
                        'offwind-float', 'ror', 'hydro']

    for path in scenario_paths:
        net = pypsa.Network(str(path))

        # Dynamically detect available electricity bus carriers for THIS network
        available_carriers = net.buses.carrier.unique()
        elec_bus_carriers = [c for c in desired_elec_carriers if c in available_carriers]

        # Calculate ELECTRICITY-ONLY supply (using bus_carrier filter)
        if elec_bus_carriers:
            elec_supply = net.statistics.supply(bus_carrier=elec_bus_carriers)
            total_elec_supply = elec_supply.sum() / 1e6  # TWh

            # RE supply (electricity only)
            re_supply = 0
            for carrier in re_carriers_comp:
                if ('Generator', carrier) in elec_supply.index:
                    re_supply += elec_supply[('Generator', carrier)]
            re_supply = re_supply / 1e6  # TWh

            # Curtailment (electricity)
            curt = net.statistics.curtailment(bus_carrier=elec_bus_carriers).sum() / 1e6
        else:
            # Fallback: use all supply if no electricity carriers found
            elec_supply = net.statistics.supply()
            total_elec_supply = elec_supply.sum() / 1e6
            re_supply = 0
            curt = net.statistics.curtailment().sum() / 1e6

        # Storage/flexibility capacity (electricity-connected)
        stor_cap = 0
        if len(net.storage_units) > 0:
            if elec_bus_carriers:
                # Filter to AC-connected storage units
                ac_su = net.storage_units[net.storage_units.bus.isin(
                    net.buses[net.buses.carrier.isin(elec_bus_carriers)].index
                )]
                stor_cap += ac_su.p_nom_opt.sum() / 1e3
            else:
                stor_cap += net.storage_units.p_nom_opt.sum() / 1e3
        bc_net = net.links[net.links.carrier == 'battery discharger'].index
        if len(bc_net) > 0:
            stor_cap += net.links.loc[bc_net, 'p_nom_opt'].sum() / 1e3

        # Prices (AC buses only)
        ac_b = net.buses[net.buses.carrier == 'AC'].index
        avg_price = 0
        if len(ac_b) > 0:
            p = net.buses_t.marginal_price[ac_b]
            avg_price = p.mean().mean()
            price_curves[path.stem] = p.mean(axis=1).sort_values(ascending=False).reset_index(drop=True)

        # CO2 from electricity generation only
        co2_total = 0
        if 'co2_emissions' in net.carriers.columns and elec_bus_carriers:
            # Filter generators connected to electricity buses
            elec_buses = net.buses[net.buses.carrier.isin(elec_bus_carriers)].index
            elec_gens = net.generators[net.generators.bus.isin(elec_buses)]
            if len(elec_gens) > 0:
                gen_out = net.generators_t.p[elec_gens.index].sum()
                gen_car = elec_gens.carrier
                co2_fac = gen_car.map(net.carriers['co2_emissions']).fillna(0)
                co2_total = (gen_out * co2_fac).sum() / 1e6

        comparison.append({
            'scenario': path.stem,
            'elec_generation_TWh': total_elec_supply,
            'system_cost_BEUR': net.objective / 1e9,
            'elec_curtailment_TWh': curt,
            'elec_re_share_%': re_supply / total_elec_supply * 100 if total_elec_supply > 0 else 0,
            'elec_flexibility_GW': stor_cap,
            'elec_avg_price_EUR_MWh': avg_price,
            'elec_co2_Mt': co2_total,
            'capex_BEUR': net.statistics.capex().sum() / 1e9,
            'opex_BEUR': net.statistics.opex().sum() / 1e9
        })

    comparison_df = pd.DataFrame(comparison).set_index('scenario')
    export_df(comparison_df, 'scenario_comparison', None)

    print("=== SCENARIO COMPARISON (Electricity Sector) ===")
    print(comparison_df.round(2))

    # Visualization
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    comparison_df['system_cost_BEUR'].plot.bar(ax=axes[0,0], color='steelblue')
    axes[0,0].set_title('System Cost (B EUR)')
    axes[0,0].set_ylabel('B EUR')

    comparison_df['elec_curtailment_TWh'].plot.bar(ax=axes[0,1], color='coral')
    axes[0,1].set_title('Electricity Curtailment (TWh)')
    axes[0,1].set_ylabel('TWh')

    comparison_df['elec_re_share_%'].plot.bar(ax=axes[0,2], color='green')
    axes[0,2].set_title('Electricity RE Share (%)')
    axes[0,2].set_ylabel('%')

    comparison_df['elec_flexibility_GW'].plot.bar(ax=axes[1,0], color='purple')
    axes[1,0].set_title('Electricity Flexibility (GW)')
    axes[1,0].set_ylabel('GW')

    comparison_df['elec_avg_price_EUR_MWh'].plot.bar(ax=axes[1,1], color='orange')
    axes[1,1].set_title('Electricity Price (EUR/MWh)')
    axes[1,1].set_ylabel('EUR/MWh')

    comparison_df['elec_co2_Mt'].plot.bar(ax=axes[1,2], color='grey')
    axes[1,2].set_title('Electricity CO2 (Mt)')
    axes[1,2].set_ylabel('Mt')

    for ax in axes.flat:
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'scenario_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Price duration curves comparison
    if len(price_curves) > 1:
        fig, ax = plt.subplots(figsize=(12, 6))
        for name, curve in price_curves.items():
            ax.plot(curve.values, label=name, lw=1.5)
        ax.set_xlabel('Hours (sorted)')
        ax.set_ylabel('EUR/MWh')
        ax.set_title('Electricity Price Duration Curves - Scenario Comparison')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.savefig(OUTPUT_DIR / 'price_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    logger.info("Only one or no networks available, skipping scenario comparison")